# 02. Qualitative Coding Exploration & Validation

**Agent 02**: Qualitative Coding (`exploration` notebook — not a production entry point)  
**Architecture Version**: `v1.1.0`  
**Active Run**: `run_20261111_002`  
**Planned Downstream Deliverable**: `themes_codebook_v1` (`data/processed/themes_codebook.json`, blocked on Agent 01 Gate 1 & Agent 07 GPU runtime)

---

### Core Analytical Invariants Demonstrated in this Notebook:
1. **Complete Question-Corpus Subtopic Discovery (`QuestionCorpusDiscoveryEngine`)**: Discovers bottom-up subtopics from the full corpus of participant responses (with documented chunk-and-consolidate preserving minority positions), never from question titles alone.
2. **Two-Stage Open Organisational Value Discovery (`ValueExtractor`)**: Stage A extracts free-form values (`original_label`, optional `polarity`) without forcing unfamiliar labels into fixed examples; Stage B proposes cross-question consolidations with uncertainty preserved for Agent 03.
3. **Distributional Preference Representation & Distinct Quantities (`ADR-0005`)**: Strictly separates `alternative_probabilities` (epistemic/mixed preference, NOT cardinal utility), `semantic_importance` (never response length), `coder_confidence`, `coder_agreement`, and `polarity`.
4. **Structured Subtopic Matching (`SubtopicMatcher`)**: Lexical Jaccard retrieves candidates only; semantic matching separates topic (`decision_object`, `mechanism`, `scope`, `definition`) from opposing positions and preserves uncertain matches for Agent 03.
5. **Selective Subtopic-Level Goods Taxonomy**: Ostrom goods labels apply at subtopic level when grounded in allocation/access/ownership/financing/risk/decision-rights evidence, overriding question-level guidance when evidence diverges.
6. **Response Status Preservation & Repair Safety**: Explicit no-opinion, insufficient-experience/information, and mixed uncertainty statuses are preserved without fabricating preferences; probability/quote repairs are strictly audited and offset-verified.

In [ ]:
import os
import sys
from pathlib import Path
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir() and (p / "data").is_dir()), Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from src.contracts import AnalyticalFrameType, MissingnessState
from src.nlp import (
    EvidenceSpanExtractor,
    NormalizedResponseInput,
    QuestionCorpusCoder,
    QuestionCorpusInput,
    SubtopicDescriptor,
    SubtopicMatcher,
    ValueExtractor,
    get_multilingual_test_fixtures,
    get_sample_question_specs,
    validate_and_repair_probabilities,
)

print("Agent 02 refined NLP Qualitative Coding engine successfully loaded.")

## 1. Multilingual Fixtures (Substantive, Mixed Uncertainty, Explicit No-Opinion, Missing)

In [ ]:
question_specs = get_sample_question_specs()
fixtures = get_multilingual_test_fixtures()

for k, v in fixtures.items():
    print(f"{k}: q={v['question_id']}, lang={v['language']}, status={v.get('response_status')}")

## 2. Two-Stage Open Organisational Value Discovery (Stage A Free-Form & Stage B Proposal)

In [ ]:
val_extractor = ValueExtractor(run_id="run_20261111_002")
span_extractor = EvidenceSpanExtractor(run_id="run_20261111_002")

unseen_fix = fixtures["EN_01"]
spans = span_extractor.extract_spans(
    participant_id=unseen_fix["participant_id"],
    question_id=unseen_fix["question_id"],
    raw_text=unseen_fix["text"],
    detected_language="en",
    quotes_or_spans=unseen_fix["expected_quotes"],
)

stage_a_val = val_extractor.extract_stage_a_value(
    participant_id=unseen_fix["participant_id"],
    question_id=unseen_fix["question_id"],
    raw_value={
        "original_label": "Mentoring Continuity",
        "value_definition": "Protecting stable doctoral and technical knowledge transfer across cohorts.",
        "polarity": None,
        "semantic_importance": 0.90,
        "coder_confidence": 0.88,
    },
    supporting_span_ids=[spans[1].span_id],
    coder_id="coder_a",
)

print("Stage A free-form label preserved:", stage_a_val.original_label)
print("Value ID:", stage_a_val.value_id, "| Alias:", stage_a_val.matched_example_alias, "| Polarity:", stage_a_val.polarity)

stage_b_proposals = val_extractor.propose_cross_question_consolidation([stage_a_val])
print("Stage B proposals (for Agent 03 reconciliation):", len(stage_b_proposals))

## 3. Structured Subtopic Matching (Topic vs. Opposing Positions)

In [ ]:
matcher = SubtopicMatcher()
d1 = SubtopicDescriptor(
    item_id="C1",
    question_id="Q06",
    label="Individual Grant Overhead Responsibility",
    definition="Who bears laboratory grant overhead financial charges within the unit.",
    decision_object="research grant overhead financial responsibility",
    mechanism="individual PI account allocation",
    scope="laboratory level",
    alternatives=["individual_pi_responsibility"],
    evidence_ids=["SPAN_1"],
)
d2 = SubtopicDescriptor(
    item_id="C2",
    question_id="Q06",
    label="Collective Grant Overhead Pooling",
    definition="Who pays laboratory grant overhead costs through pooled collective funds.",
    decision_object="research grant overhead financial responsibility",
    mechanism="collective pooled mutualized budget",
    scope="laboratory level",
    alternatives=["collective_mutualized_responsibility"],
    evidence_ids=["SPAN_2"],
)
proposal = matcher.evaluate_pair(d1, d2)
print("Match decision:", proposal.match_decision, "| Lex score:", proposal.lexical_retrieval_score)

## 4. Probability Repair Audit & Full Question-Corpus Entry Point (`QuestionCorpusCoder`)

In [ ]:
dist, repair_rec = validate_and_repair_probabilities({"option_a": 75.0, "option_b": 25.0})
print("Repaired percentage distribution:", dist, "| Audit:", repair_rec.repair_action)
print("Canonical coding entry point:", QuestionCorpusCoder.code_question_corpus.__qualname__)